In [4]:
import sys
import json

# from qdrant_client import QdrantClient
from pathlib import Path
from tqdm.asyncio import tqdm

# Add parent directory to path to import from implementation package
# Notebooks are in implementation/notebooks/, so we go up two levels to project root
sys.path.insert(0, str(Path().resolve().parent.parent))

from implementation.classes.movie import BaseMovie
from db.redis import init_redis, close_redis
from db.trending_movies import refresh_trending_scores
from db.ingest_movie import ingest_movie, ingest_movies_to_qdrant_batched
from db.postgres import (
    pool, 
    refresh_title_token_doc_frequency,
    refresh_movie_popularity_scores,
    batch_upsert_genre_dictionary,
    batch_upsert_language_dictionary, 
    batch_upsert_maturity_dictionary,
    batch_upsert_watch_method_dictionary
)

In [5]:
# LOAD MOVIES

json_path = Path("../../saved_imdb_movies.json")
with open(json_path, "r", encoding="utf-8") as f:
    movies_data = json.load(f)

# Convert each dictionary to an IMDBMovie object
movies = [BaseMovie(**movie_dict) for movie_dict in movies_data]

In [6]:
# Open the pool and establish initial connections
await pool.open()

# First batch insert anything that's hard-coded
print("Upserting hard-coded")
await batch_upsert_genre_dictionary()
await batch_upsert_language_dictionary()
await batch_upsert_maturity_dictionary()
await batch_upsert_watch_method_dictionary()

# Run ingest_movie on every movie in parallel; tqdm.gather shows progress without blocking async
print("Ingesting movies (Postgres)")
_ = await tqdm.gather(*[ingest_movie(movie) for movie in movies], desc="Ingesting movies (Postgres)")

# Postgres materialization
print("Refreshing materialized views")
await refresh_title_token_doc_frequency()
await refresh_movie_popularity_scores()

# Vector search
print("Ingesting movies to Qdrant")
await ingest_movies_to_qdrant_batched(movies)

# Redis
print("Refreshing trending scores")
await init_redis()
await refresh_trending_scores()

# Cleanup
print("Closing connections")
await pool.close()
await close_redis()

Upserting hard-coded
Ingesting movies (Postgres)


Ingesting movies (Postgres): 100%|██████████| 50/50 [00:00<00:00, 136.68it/s]


Refreshing materialized views
Ingesting movies to Qdrant
Refreshing trending scores
pages_needed: 25


Closing connections
